# Chương 2: Mạng Nơ-ron Tích Chập - MNIST và Fashion-MNIST

## Mục tiêu
Hiểu cách CNN trích xuất đặc trưng với 2 datasets đơn giản: MNIST và Fashion-MNIST.

## Datasets
1. MNIST: Handwritten digits
2. Fashion-MNIST: Fashion items

In [13]:
# Import Required Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## Dataset 1: MNIST

### Load và Visualize Data

In [14]:
# Load MNIST Dataset
(x_train_mnist, y_train_mnist), (x_test_mnist, y_test_mnist) = keras.datasets.mnist.load_data()

print("MNIST Dataset:")
print(f"Training data shape: {x_train_mnist.shape}")
print(f"Test data shape: {x_test_mnist.shape}")
print(f"Number of classes: {len(np.unique(y_train_mnist))}")

# Normalize data
x_train_mnist = x_train_mnist.astype("float32") / 255
x_test_mnist = x_test_mnist.astype("float32") / 255

# Reshape for CNN
x_train_mnist = np.expand_dims(x_train_mnist, -1)
x_test_mnist = np.expand_dims(x_test_mnist, -1)

print(f"Reshaped training data: {x_train_mnist.shape}")

MNIST Dataset:
Training data shape: (60000, 28, 28)
Test data shape: (10000, 28, 28)
Number of classes: 10
Reshaped training data: (60000, 28, 28, 1)


In [15]:
# Plot 1: Sample Images
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i+1)
    plt.imshow(x_train_mnist[i].squeeze(), cmap='gray')
    plt.title(f'Label: {y_train_mnist[i]}')
    plt.axis('off')
plt.tight_layout()
plt.savefig('mnist_sample_images.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot 2: Class Distribution
plt.figure(figsize=(10, 6))
unique, counts = np.unique(y_train_mnist, return_counts=True)
plt.bar(unique, counts)
plt.title('Class Distribution in MNIST Training Set')
plt.xlabel('Digit')
plt.ylabel('Count')
plt.xticks(unique)
plt.savefig('mnist_class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot 3: Average Pixel Intensity per Class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for digit in range(10):
    ax = axes[digit//5, digit%5]
    digit_images = x_train_mnist[y_train_mnist == digit]
    avg_image = np.mean(digit_images, axis=0).squeeze()
    ax.imshow(avg_image, cmap='gray')
    ax.set_title(f'Avg Digit {digit}')
    ax.axis('off')
plt.tight_layout()
plt.savefig('mnist_average_digits.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot 4: Pixel Intensity Distribution
plt.figure(figsize=(10, 6))
plt.hist(x_train_mnist.flatten(), bins=50, alpha=0.7)
plt.title('Pixel Intensity Distribution')
plt.xlabel('Pixel Value')
plt.ylabel('Frequency')
plt.savefig('mnist_pixel_intensity_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

### Build và Train Simple CNN Model

In [16]:
# Build Simple CNN Model for MNIST
def create_simple_cnn():
    model = keras.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    return model

model_mnist = create_simple_cnn()
model_mnist.compile(optimizer='adam',
                   loss='sparse_categorical_crossentropy',
                   metrics=['accuracy'])

model_mnist.summary()

Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_7 (Conv2D)           (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d_4 (MaxPoolin  (None, 13, 13, 32)        0         
 g2D)                                                            
                                                                 
 conv2d_8 (Conv2D)           (None, 11, 11, 64)        18496     
                                                                 
 max_pooling2d_5 (MaxPoolin  (None, 5, 5, 64)          0         
 g2D)                                                            
                                                                 
 flatten_4 (Flatten)         (None, 1600)              0         
                                                                 
 dense_9 (Dense)             (None, 64)               

In [17]:
# Train Simple CNN Model for MNIST
history_mnist = model_mnist.fit(x_train_mnist, y_train_mnist,
                               epochs=5, batch_size=128,
                               validation_split=0.1, verbose=1)

Epoch 1/5


422/422 [==============================] - 3s 6ms/step - loss: 0.2503 - accuracy: 0.9269 - val_loss: 0.0948 - val_accuracy: 0.9755
Epoch 2/5
422/422 [==============================] - 2s 6ms/step - loss: 0.0665 - accuracy: 0.9798 - val_loss: 0.0520 - val_accuracy: 0.9858
Epoch 3/5
422/422 [==============================] - 2s 6ms/step - loss: 0.0469 - accuracy: 0.9862 - val_loss: 0.0459 - val_accuracy: 0.9883
Epoch 4/5
422/422 [==============================] - 2s 5ms/step - loss: 0.0362 - accuracy: 0.9886 - val_loss: 0.0383 - val_accuracy: 0.9902
Epoch 5/5
422/422 [==============================] - 2s 5ms/step - loss: 0.0280 - accuracy: 0.9911 - val_loss: 0.0367 - val_accuracy: 0.9892


In [18]:
# Build LeNet-5 Model for MNIST
def create_lenet5():
    model = keras.Sequential([
        layers.Conv2D(6, (5, 5), activation='tanh', input_shape=(28, 28, 1)),
        layers.AveragePooling2D((2, 2)),
        layers.Conv2D(16, (5, 5), activation='tanh'),
        layers.AveragePooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(120, activation='tanh'),
        layers.Dense(84, activation='tanh'),
        layers.Dense(10, activation='softmax')
    ])
    return model

model_mnist_2 = create_lenet5()
model_mnist_2.compile(optimizer='adam',
                     loss='sparse_categorical_crossentropy',
                     metrics=['accuracy'])

# Train LeNet-5 Model for MNIST
history_mnist_2 = model_mnist_2.fit(x_train_mnist, y_train_mnist,
                                   epochs=5, batch_size=128,
                                   validation_split=0.1, verbose=1)

Epoch 1/5
422/422 [==============================] - 2s 4ms/step - loss: 0.3802 - accuracy: 0.8922 - val_loss: 0.1471 - val_accuracy: 0.9602
Epoch 2/5
422/422 [==============================] - 2s 4ms/step - loss: 0.1438 - accuracy: 0.9565 - val_loss: 0.0942 - val_accuracy: 0.9735
Epoch 3/5
422/422 [==============================] - 2s 4ms/step - loss: 0.0937 - accuracy: 0.9713 - val_loss: 0.0746 - val_accuracy: 0.9788
Epoch 4/5
422/422 [==============================] - 2s 4ms/step - loss: 0.0693 - accuracy: 0.9788 - val_loss: 0.0616 - val_accuracy: 0.9825
Epoch 5/5
422/422 [==============================] - 2s 4ms/step - loss: 0.0549 - accuracy: 0.9831 - val_loss: 0.0668 - val_accuracy: 0.9777


In [19]:

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_mnist.history['accuracy'], label='Simple CNN Train')
plt.plot(history_mnist_2.history['accuracy'], label='LeNet-5 Train')
plt.title('MNIST Accuracy Comparison')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history_mnist.history['loss'], label='Simple CNN Loss')
plt.plot(history_mnist_2.history['loss'], label='LeNet-5 Loss')
plt.title('MNIST Loss Comparison')
plt.legend()
plt.savefig('plots/ch2_mnist_history.png', dpi=300)
plt.show()


In [20]:
# Confusion Matrix
y_pred_mnist = np.argmax(model_mnist.predict(x_test_mnist), axis=1)
cm = confusion_matrix(y_test_mnist, y_pred_mnist)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.title('MNIST - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.savefig('mnist_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Feature Map Visualization
# Get the first conv layer
conv_layer = model_mnist.layers[0]
feature_model = keras.Model(inputs=model_mnist.input, outputs=conv_layer.output)

# Select a sample image
sample_image = x_test_mnist[0:1]
feature_maps = feature_model.predict(sample_image)

# Plot first 16 feature maps
plt.figure(figsize=(12, 8))
for i in range(16):
    plt.subplot(4, 4, i+1)
    plt.imshow(feature_maps[0, :, :, i], cmap='viridis')
    plt.axis('off')
plt.suptitle('Feature Maps from First Convolutional Layer')
plt.savefig('mnist_feature_maps.png', dpi=300, bbox_inches='tight')
plt.show()

 39/313 [==>...........................] - ETA: 0s

1/1 [==============================] - 0s 18ms/step


## Dataset 2: Fashion-MNIST

### Load và Visualize Data

In [21]:
# Load Fashion-MNIST Dataset
(x_train_fashion, y_train_fashion), (x_test_fashion, y_test_fashion) = keras.datasets.fashion_mnist.load_data()

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print("Fashion-MNIST Dataset:")
print(f"Training data shape: {x_train_fashion.shape}")
print(f"Test data shape: {x_test_fashion.shape}")

# Normalize and reshape
x_train_fashion = x_train_fashion.astype("float32") / 255
x_test_fashion = x_test_fashion.astype("float32") / 255
x_train_fashion = np.expand_dims(x_train_fashion, -1)
x_test_fashion = np.expand_dims(x_test_fashion, -1)

Fashion-MNIST Dataset:
Training data shape: (60000, 28, 28)
Test data shape: (10000, 28, 28)


In [22]:
# === Fashion-MNIST: Model 1 (Simple CNN) + Model 2 (Deep CNN) ===
model_fashion_1 = keras.Sequential([
    keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(10, activation='softmax')
])
model_fashion_1.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_fashion_1 = model_fashion_1.fit(x_train_fashion, y_train_fashion,
                                         batch_size=128, epochs=5, validation_split=0.1, verbose=0)

model_fashion_2 = keras.Sequential([
    keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(32, (3,3), activation='relu'),
    keras.layers.MaxPooling2D((2,2)),
    keras.layers.Dropout(0.25),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(10, activation='softmax')
])
model_fashion_2.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_fashion_2 = model_fashion_2.fit(x_train_fashion, y_train_fashion,
                                         batch_size=128, epochs=5, validation_split=0.1, verbose=0)
print("Simple CNN Fashion acc:", history_fashion_1.history['val_accuracy'][-1])
print("Deep CNN Fashion acc:",   history_fashion_2.history['val_accuracy'][-1])

# Plot 4: Fashion-MNIST Accuracy/Loss comparison
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_fashion_1.history['accuracy'], label='Simple CNN')
axes[0].plot(history_fashion_2.history['accuracy'], label='Deep CNN')
axes[0].set_title('Fashion-MNIST Accuracy'); axes[0].legend()
axes[1].plot(history_fashion_1.history['loss'], label='Simple CNN')
axes[1].plot(history_fashion_2.history['loss'], label='Deep CNN')
axes[1].set_title('Fashion-MNIST Loss'); axes[1].legend()
plt.tight_layout()
plt.savefig('plots/ch2_fashion_history.png', dpi=300)
plt.close()
print("Saved: plots/ch2_fashion_history.png")


Simple CNN Fashion acc: 0.9066666960716248
Deep CNN Fashion acc: 0.9158333539962769
Saved: plots/ch2_fashion_history.png


In [23]:
# Plot 1: Sample Images
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i+1)
    plt.imshow(x_train_fashion[i].squeeze(), cmap='gray')
    plt.title(class_names[y_train_fashion[i]], fontsize=8)
    plt.axis('off')
plt.tight_layout()
plt.savefig('fashion_mnist_sample_images.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot 2: Class Distribution
plt.figure(figsize=(12, 6))
unique, counts = np.unique(y_train_fashion, return_counts=True)
plt.bar(range(10), counts)
plt.title('Class Distribution in Fashion-MNIST Training Set')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(range(10), class_names, rotation=45)
plt.savefig('fashion_mnist_class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot 3: Average Images per Class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for item in range(10):
    ax = axes[item//5, item%5]
    item_images = x_train_fashion[y_train_fashion == item]
    avg_image = np.mean(item_images, axis=0).squeeze()
    ax.imshow(avg_image, cmap='gray')
    ax.set_title(class_names[item], fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.savefig('fashion_mnist_average_items.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot 4: Pixel Intensity Distribution
plt.figure(figsize=(10, 6))
plt.hist(x_train_fashion.flatten(), bins=50, alpha=0.7)
plt.title('Pixel Intensity Distribution - Fashion-MNIST')
plt.xlabel('Pixel Value')
plt.ylabel('Frequency')
plt.savefig('fashion_mnist_pixel_intensity_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [24]:
# Create directory for saved plots
import os
if not os.path.exists('plots'):
    os.makedirs('plots')

# Move all saved PNG files to plots directory
import shutil
png_files = [f for f in os.listdir('.') if f.endswith('.png')]
for file in png_files:
    shutil.move(file, os.path.join('plots', file))

print("All plots saved to 'plots' directory for report writing.")

All plots saved to 'plots' directory for report writing.
